In [ ]:
#OBSOLETE this is the clean version, see test_proj.ipynb for useful version
#Grok prompt: 
#python geemap transform a bounding box from WGS to UTM8. Then regularize so it is a rectangle in UTM8. Then transform back to WGS.
#turn this into a function with wgs84_bbox as input and wgs84_bbox_new as output
#save shapely.geometry.polygon.Polygon to csv and reload
#store all coordinates on one row of the data frame and csv

import geemap
import ee
import pyproj
from shapely.geometry import box, Polygon
from shapely.ops import transform
import pandas as pd

In [ ]:
def transform_regularize_bbox(wgs84_bbox):
    """
    Transform a WGS84 bounding box to UTM Zone 8N, regularize it to a rectangle,
    and transform it back to WGS84, returning a Shapely Polygon.
    
    Args:
        wgs84_bbox (list or tuple): [minx, miny, maxx, maxy] in WGS84 (lon, lat)
    
    Returns:
        shapely.geometry.polygon.Polygon: Regularized polygon in WGS84
    """
    # Define CRS for WGS84 and UTM Zone 8N
    wgs84 = pyproj.CRS("EPSG:4326")
    utm8 = pyproj.CRS("EPSG:32608")
    
    # Create transformers
    project_to_utm = pyproj.Transformer.from_crs(wgs84, utm8, always_xy=True).transform
    project_to_wgs84 = pyproj.Transformer.from_crs(utm8, wgs84, always_xy=True).transform

    # Convert WGS84 bbox to Shapely geometry
    wgs84_box = box(wgs84_bbox[0], wgs84_bbox[1], wgs84_bbox[2], wgs84_bbox[3])

    # Transform to UTM Zone 8N and regularize
    utm8_box = transform(project_to_utm, wgs84_box)
    utm8_rect = utm8_box.envelope

    # Transform back to WGS84
    wgs84_rect = transform(project_to_wgs84, utm8_rect)
    
    return wgs84_rect

def save_polygon_to_csv(polygon, filename):
    """
    Save a Shapely Polygon to a CSV file with all exterior coordinates in a single row.
    
    Args:
        polygon (shapely.geometry.polygon.Polygon): The polygon to save
        filename (str): Path to the output CSV file
    """
    # Extract exterior coordinates
    coords = list(polygon.exterior.coords)
    # Flatten coordinates into a single row with columns x1, y1, x2, y2, ...
    coord_dict = {}
    for i, (x, y) in enumerate(coords, 1):
        coord_dict[f'x{i}'] = x
        coord_dict[f'y{i}'] = y
    # Create a DataFrame with one row
    df = pd.DataFrame([coord_dict])
    # Save to CSV
    df.to_csv(filename, index=False)
    print(f"Polygon saved to {filename}")

def load_polygon_from_csv(filename):
    """
    Load a Shapely Polygon from a CSV file with coordinates in a single row.
    
    Args:
        filename (str): Path to the input CSV file
    
    Returns:
        shapely.geometry.polygon.Polygon: Reconstructed polygon
    """
    # Read CSV
    df = pd.read_csv(filename)
    # Extract coordinates from the single row
    coords = []
    i = 1
    while f'x{i}' in df.columns and f'y{i}' in df.columns:
        x = df[f'x{i}'].iloc[0]
        y = df[f'y{i}'].iloc[0]
        coords.append((x, y))
        i += 1
    # Ensure the polygon is closed (first and last points are the same)
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])
    return Polygon(coords)

In [ ]:
# Example usage
wgs84_bbox = [-135.0, 57.0, -134.0, 58.0]
print("Original WGS84 BBox:", wgs84_bbox)

# Get the regularized polygon
wgs84_polygon = transform_regularize_bbox(wgs84_bbox)
print("Regularized WGS84 BBox (bounds):", wgs84_polygon.bounds)
print("Regularized WGS84 BBox:", wgs84_polygon)

# Save the polygon to CSV
csv_filename = "regularized_polygon.csv"
save_polygon_to_csv(wgs84_polygon, csv_filename)

# Reload the polygon from CSV
reloaded_polygon = load_polygon_from_csv(csv_filename)
print("Reloaded Polygon (bounds):", reloaded_polygon.bounds)
print("Reloaded Polygon:", reloaded_polygon)

# Verify the reloaded polygon matches the original
print("Polygons match:", wgs84_polygon.equals(reloaded_polygon))